In [ ]:
from os import getcwd
cwd_os = getcwd()
print(f"working dir: {cwd_os}")


single generation turn

In [1]:
# https://proproprogs.ru/ga/ga-delaem-geneticheskiy-algoritm-dlya-zadachi-onemax

from sqlite3 import connect
from os import path
from random import seed, random, randint, shuffle


POPULATION_SIZE = 200
P_CROSSOVER = 0.9
P_MUTATION = 0.1
RANDOM_SEED = 42
seed(RANDOM_SEED)

db_path = '../../data/train.db'
if not path.exists(db_path):
	raise Exception("db not exists")

with connect(db_path, autocommit=True) as conn:
	cursor = conn.cursor()

	cursor.execute(f'''
		select s.id, s.fitness, w.joint, w.range
		from (
			select s.id, s.fitness
			from session s
			where s.fitness > 0
			order by s.fitness desc --s.ctime desc, s.id
			limit {POPULATION_SIZE}
		) s
		join walk_param w on s.id = w.session_id		
	''')

	sessions = []
	session_id = -1
	for id, fitness, joint, range_ in cursor.fetchall():
		if session_id != id:
			session = { "id": id, "fitness": fitness, "params": { joint: range_ } }
			session_id = id
			sessions.append(session)
		else:
			session["params"][joint] = range_

	pop_size = len(sessions)
	if pop_size % 2 > 0:
		sessions.pop()
		pop_size -= 1
	print(sessions)

	max_pop_id = pop_size - 1
	print(max_pop_id)

	param_keys = list(sessions[0]["params"].keys())
	print(param_keys)

	max_params = len(param_keys) - 1
	print(max_params)
	
	max_fitness = max(map(lambda x: x["fitness"], sessions))
	print(max_fitness)


	# selection
	offspring = []
	for _ in range(pop_size):
		i1 = i2 = i3 = 0
		while i1 == i2 or i1 == i3 or i2 == i3:
			i1, i2, i3 = randint(0, max_pop_id), randint(0, max_pop_id), randint(0, max_pop_id)
		offspring.append(max([sessions[i1], sessions[i2], sessions[i3]], key=lambda ind: ind["fitness"]))
	print(offspring)


	# crossover
	for i in range(0, pop_size, 2):
		if random() < P_CROSSOVER:
			child1 = offspring[i]
			child2 = offspring[i + 1]
			# print(i)
			# print(child1, child2)
			joint = param_keys[randint(0, max_params)]
			child1["params"][joint], child2["params"][joint] = child2["params"][joint], child1["params"][joint]
			# print(child1, child2)


	# mutation
	for mutant in offspring:
		if random() < P_MUTATION:
			joint = param_keys[randint(0, max_params)]
			mutant["params"][joint] = random()


	# save to DB
	for session in offspring:
		cursor.execute("INSERT INTO session (fitness) VALUES (null)")
		session_id = cursor.lastrowid
		for joint, range_ in dict(session["params"]).items():
			cursor.execute("INSERT INTO walk_param (session_id, joint, range) VALUES (?, ?, ?)", (session_id, joint, range_))





[{'id': 7, 'fitness': 0.5126073271565798, 'params': {'foot': 0.1870436709129503, 'hip_L': 0.18881921051650452, 'calf_L': 0.6485882277751974, 'hip_R': 0.9884751834528, 'calf_R': 0.09159688565665801}}, {'id': 8, 'fitness': 0.5125573588196924, 'params': {'foot': 0.14494694809354788, 'hip_L': 0.1385981780449312, 'calf_L': 0.60339555944784, 'hip_R': 0.8969889423430474, 'calf_R': 0.06004657185352681}}, {'id': 10, 'fitness': 0.515120388247905, 'params': {'foot': 0.19997225478278188, 'hip_L': 0.08389403094974174, 'calf_L': 0.4257067389828273, 'hip_R': 0.92842568634531, 'calf_R': 0.00800365963015017}}, {'id': 13, 'fitness': 0.507566922228342, 'params': {'foot': 0.18234176236926275, 'hip_L': 0.08627422451251163, 'calf_L': 0.5719874847245121, 'hip_R': 0.9756650723235115, 'calf_R': 0.18262933436467965}}, {'id': 17, 'fitness': 0.5100836325993666, 'params': {'foot': 0.2889743070750851, 'hip_L': 0.17175031905046023, 'calf_L': 0.3144544499957398, 'hip_R': 0.9076949185250108, 'calf_R': 0.16820570000951